# Eye model explorer

Move the sliders to see how each parameter changes the rendered mask.

- **Left:** the render (model classes: lid/skin, sclera, iris, pupil, caruncle).
- **Middle:** a reference eye from the dataset, with the render's outlines (solid) and the
  annotation (dashed). White = sclera, cyan = iris, magenta = caruncle.
- **Right:** the annotation.
- **Below the plots:** the eye proportions of the render and of the annotation. *Fissure / iris*
  is the width of the visible eye over the iris width. *Opening / iris* is the eye's height at
  the iris centre over the iris width.

Buttons:
- **Defaults** resets the sliders.
- **Network prediction** sets the sliders to the checkpoint's prediction for the reference
  eye. Parameters the checkpoint keeps fixed take their fixed value.
- **Beyond bounds** stretches each slider 50% past the training bounds (shaded in orange) to
  try new limits.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

from eyemodel3d import train as T
import eyemodel3d.renderer.eye_model as em

warnings.filterwarnings("ignore", "Mean of empty slice")  # eyes with no foreground at all
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import ipywidgets as wd
from IPython.display import display

# ---- edit these
CKPT = ROOT / "runs/combined_zoom1/best.pt"  # None: no reference images / predictions
DATA = None  # dataset folder; None = the checkpoint's
SIZE = (128, 160)  # (H, W) of the render; taken from the checkpoint when there is one

In [ ]:
if CKPT:
    net, space, fixed, meta = T.load_model(CKPT, dev)
    SIZE = tuple(meta["size"])
    pairs = T.list_pairs(DATA or meta["data"])
    val = set(meta["val"])
    ref_ds = T.EyeMaskDataset([p for p in pairs if p[0].name in val] or pairs, *SIZE)
    print(f"{CKPT.name}: epoch {meta['epoch']}, val Dice {meta['val_dice']:.3f}; "
          f"{len(ref_ds)} reference eyes")
else:
    net, ref_ds = None, None
H, W = SIZE

# Slider ranges: the training bounds (f and the camera shift depend on the image size)
bounds = {n: tuple(s[1:3]) for n, s in em.PARAM_SPECS.items()}
bounds.update({n: tuple(b[:2]) for n, b in T.default_bounds(W, 6.0).items()})
start = dict(em.DEFAULTS, f=T.default_focal(W))
if CKPT:
    start.update(meta["fixed"])
    bounds.update({n: tuple(b[:2]) for n, b in meta["bounds"].items()})
start.pop("medial", None)
GROUPS = {"eye": [], "cam": [], "lids": []}
for n, s in em.PARAM_SPECS.items():
    GROUPS[s[0]].append(n)

In [ ]:
def render(vals, medial, tau):
    leaves = em.make_leaves(**vals, medial=medial)
    with torch.no_grad():
        return em.render(*em.split_params(leaves), H=H, W=W, tau=tau)  # (H, W, 5)


def proportions(lbl):
    """(fissure width, opening at the iris centre) / iris width, from target labels."""
    iris, eye = lbl == 2, (lbl == 1) | (lbl == 2) | (lbl == 3)
    if iris.sum() < 10 or eye.sum() < 10:
        return np.nan, np.nan
    xs = np.nonzero(iris.any(0))[0]
    wi = xs.max() - xs.min() + 1
    xe = np.nonzero(eye.any(0))[0]
    col = np.nonzero(eye[:, int(round(np.nonzero(iris)[1].mean()))])[0]
    opening = col.max() - col.min() + 1 if len(col) else np.nan
    return (xe.max() - xe.min() + 1) / wi, opening / wi


def outlines(ax, lbl, ls):
    for c, col in ((1, "w"), (2, "c"), (3, "m")):
        m = (lbl == c).astype(float)
        if m.any() and not m.all():
            ax.contour(m, [0.5], colors=col, linewidths=1, linestyles=ls)

In [ ]:
sliders = {}
for n in em.PARAM_SPECS:
    lo, hi = bounds[n]
    step = 10 ** (np.floor(np.log10(hi - lo)) - 2)
    sliders[n] = wd.FloatSlider(
        value=float(np.clip(start[n], lo, hi)), min=lo, max=hi, step=step, description=n,
        continuous_update=False, readout_format=".2f" if step < 0.1 else ".1f",
        style={"description_width": "70px"}, layout=wd.Layout(width="360px"),
    )
medial_w = wd.ToggleButtons(options=[("OD (caruncle right)", 1), ("OS (caruncle left)", -1)],
                            description="laterality")
tau_w = wd.FloatLogSlider(value=0.05, base=10, min=-2.5, max=0, step=0.05, description="tau (mm)",
                          continuous_update=False)
ref_w = wd.IntSlider(0, 0, max(0, len(ref_ds) - 1) if ref_ds else 0, description="reference",
                     continuous_update=False, disabled=ref_ds is None)
beyond_w = wd.Checkbox(False, description="beyond bounds")
b_default = wd.Button(description="Defaults")
b_pred = wd.Button(description="Network prediction", disabled=net is None)
out = wd.Output()
info = wd.HTML()
ref_cache = {}


def get_ref(i):
    if i not in ref_cache:
        x, y, med, name = ref_ds[i]
        ref_cache[i] = (x, y.numpy(), float(med), name)
    return ref_cache[i]


def update(*_):
    vals = {n: s.value for n, s in sliders.items()}
    probs = render(vals, medial_w.value, tau_w.value)
    target = torch.zeros(*probs.shape[:2], T.NT)
    target.index_add_(2, torch.tensor(T.MODEL_TO_TARGET), probs)
    lbl = target.argmax(-1).numpy()
    fr, op = proportions(lbl)
    rows = [f"<b>render</b>: fissure / iris {fr:.2f}, opening / iris {op:.2f}"]
    with out:
        out.clear_output(wait=True)
        n = 3 if ref_ds else 1
        fig, axs = plt.subplots(1, n, figsize=(4.2 * n, 3.6), squeeze=False)
        axs = axs[0]
        axs[0].imshow(em.to_rgb(probs))
        axs[0].set_title("render")
        if ref_ds:
            x, y, med, name = get_ref(ref_w.value)
            img = (x * T.IMAGENET_STD + T.IMAGENET_MEAN).clamp(0, 1).permute(1, 2, 0).numpy()
            axs[1].imshow(img)
            outlines(axs[1], lbl, "solid")
            outlines(axs[1], np.where(y == T.IGNORE, 0, y), "dashed")
            axs[1].set_title(f"{name}  ({'OD' if med > 0 else 'OS'})", fontsize=9)
            axs[2].imshow(T.colorize(y))
            axs[2].set_title("annotation")
            fa, oa = proportions(y)
            d = T.hard_dice(target[None], torch.from_numpy(y)[None])[0]
            rows.append(f"<b>annotation</b>: fissure / iris {fa:.2f}, opening / iris {oa:.2f}")
            rows.append("Dice " + ", ".join(f"{c} {v:.3f}" for c, v in
                                            zip(T.TARGET_CLASSES[1:], d[1:].tolist())))
        for a in axs:
            a.set_axis_off()
        plt.tight_layout()
        plt.show()
    info.value = "<br>".join(rows)


def set_ranges(*_):
    k = 0.5 if beyond_w.value else 0.0
    for n, s in sliders.items():
        lo, hi = bounds[n]
        span = hi - lo
        v = s.value
        s.min, s.max = -1e9, 1e9  # avoid min > max while changing both
        s.min, s.max = lo - k * span, hi + k * span
        s.value = float(np.clip(v, s.min, s.max))
        s.style.handle_color = None


def outside_marks(*_):
    for n, s in sliders.items():
        lo, hi = bounds[n]
        s.style.handle_color = "orange" if not lo <= s.value <= hi else None


def set_values(vals, medial=None):
    for s in sliders.values():
        s.unobserve(update, "value")
    for n, v in vals.items():
        if n in sliders:
            sliders[n].value = float(np.clip(v, sliders[n].min, sliders[n].max))
    if medial is not None:
        medial_w.value = medial
    for s in sliders.values():
        s.observe(update, "value")
    outside_marks()
    update()


def on_default(_):
    set_values({n: start[n] for n in sliders})


def on_pred(_):
    x, y, med, name = get_ref(ref_w.value)
    with torch.no_grad():
        z, lat = net(x[None].to(dev))
        v = space.values(z)[0].cpu()
    vals = dict(meta["fixed"])
    vals.update({n: float(v[i]) for i, n in enumerate(space.names)})
    medial = int(vals.pop("medial")) if "medial" in vals else (1 if lat.item() > 0 else -1)
    set_values(vals, medial)


for s in sliders.values():
    s.observe(update, "value")
    s.observe(outside_marks, "value")
for w in (medial_w, tau_w, ref_w):
    w.observe(update, "value")
beyond_w.observe(set_ranges, "value")
b_default.on_click(on_default)
b_pred.on_click(on_pred)

tabs = wd.Tab([wd.VBox([sliders[n] for n in GROUPS[g]]) for g in GROUPS])
for i, g in enumerate(["eye", "camera", "lids + caruncle"]):
    tabs.set_title(i, g)
controls = wd.VBox([wd.HBox([b_default, b_pred, beyond_w]), medial_w, tau_w, ref_w, tabs])
display(wd.HBox([controls, wd.VBox([out, info])]))
update()

## Parameter sweep

Renders of one parameter across its range, all others at the current slider values.
Change `PARAM` and rerun.

In [ ]:
PARAM = "open_up"
N = 7

lo, hi = sliders[PARAM].min, sliders[PARAM].max
vals = {n: s.value for n, s in sliders.items()}
fig, axs = plt.subplots(1, N, figsize=(2.4 * N, 2.3))
for a, v in zip(axs, np.linspace(lo, hi, N)):
    a.imshow(em.to_rgb(render(dict(vals, **{PARAM: v}), medial_w.value, tau_w.value)))
    a.set_title(f"{PARAM} = {v:.2f}", fontsize=9)
    a.set_axis_off()
plt.tight_layout()